In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import pickle

# Compare models only for Temperature
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=30,
        random_state=42,
        n_jobs=-1
    )
}

scores = {}

print("\nTEMPERATURE MODEL COMPARISON")
print("=" * 45)

for name, model in models.items():
    model.fit(train[features], train["Temperature"])
    predictions = model.predict(test[features])

    mae = mean_absolute_error(
        test["Temperature"], predictions
    )
    scores[name] = mae

    print(f"{name}: MAE = {mae:.3f} °C")

# Select the model with the lowest Temperature MAE
best_model_name = min(scores, key=scores.get)
best_model = models[best_model_name]

print("=" * 45)
print("Best Model for Temperature:", best_model_name)
print(f"Lowest Temperature MAE: {scores[best_model_name]:.3f} °C")

# Train the selected temperature model on all historical data
best_model.fit(data[features], data["Temperature"])

# Save the selected temperature model and its MAE
with open("/content/best_temperature_model.pkl", "wb") as file:
    pickle.dump({
        "model": best_model,
        "features": features,
        "mae": scores[best_model_name]
    }, file)





TEMPERATURE MODEL COMPARISON
Linear Regression: MAE = 1.485 °C
Decision Tree: MAE = 2.079 °C
Random Forest: MAE = 1.479 °C
Best Model for Temperature: Random Forest
Lowest Temperature MAE: 1.479 °C


In [1]:

import pandas as pd
import pickle
from datetime import datetime

# 1. Load the best temperature model
with open("/content/best_temperature_model.pkl", "rb") as file:
    saved_model = pickle.load(file)

temperature_model = saved_model["model"]
features = saved_model["features"]
mae = saved_model["mae"]

# 2. Load and prepare historical weather data
df = pd.read_csv("/content/weather.csv")
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

df = df.dropna(subset=["Date"])
df = df.sort_values("Date").drop_duplicates("Date")
df = df.set_index("Date").asfreq("D")

targets = ["Temperature", "Humidity", "Rainfall"]

for col in targets:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[targets] = df[targets].interpolate().ffill().bfill()
history = df[targets].copy()

# 3. Ask for the forecast date
date_text = input("Enter date (DD.MM.YYYY): ")

try:
    target_date = pd.Timestamp(
        datetime.strptime(date_text, "%d.%m.%Y")
    )
except ValueError:
    print("Invalid date. Use DD.MM.YYYY.")
    raise SystemExit

last_date = history.index[-1]

if target_date <= last_date:
    print("Enter a date after", last_date.strftime("%d.%m.%Y"))
    raise SystemExit

# 4. Load the saved humidity and rainfall models
# These models must have been saved separately in Part 1.
with open("/content/weather_models.pkl", "rb") as file:
    other_saved_data = pickle.load(file)

other_models = other_saved_data["models"]

# 5. Predict day by day until the requested date
current_date = last_date + pd.Timedelta(days=1)
result = None

print("Calculating forecast...")

while current_date <= target_date:
    row = {
        "Month": current_date.month,
        "Day": current_date.day,
        "DayOfYear": current_date.dayofyear,
        "Year": current_date.year
    }

    for col in targets:
        for lag in range(1, 8):
            row[f"{col}_lag{lag}"] = history[col].iloc[-lag]

    X = pd.DataFrame([row], columns=features)

    result = {
        "Temperature": float(
            temperature_model.predict(X)[0]
        ),
        "Humidity": float(
            other_models["Humidity"].predict(X)[0]
        ),
        "Rainfall": float(
            other_models["Rainfall"].predict(X)[0]
        )
    }

    history.loc[current_date, targets] = [
        result[col] for col in targets
    ]

    current_date += pd.Timedelta(days=1)

# 6. Calculate approximate temperature range
temperature = result["Temperature"]
lower = temperature - 1.96 * mae
upper = temperature + 1.96 * mae

# 7. Display final forecast
print("\n" + "=" * 50)
print("       PROBABILISTIC WEATHER FORECAST")
print("=" * 50)

print("Date                  :", target_date.strftime("%Y-%m-%d"))
print(f"Best Temperature Model: {type(temperature_model).__name__}")
print(f"Predicted Temperature : {temperature:.2f} °C")
print(f"Possible Range        : {lower:.2f} - {upper:.2f} °C")
print(f"Predicted Humidity    : {result['Humidity']:.2f} %")
print(f"Predicted Rainfall    : {result['Rainfall']:.2f} mm")

print("=" * 50)


FileNotFoundError: [Errno 2] No such file or directory: '/content/best_temperature_model.pkl'